# Exportação da Camada Gold para CSV — Power BI

Exporta todas as tabelas da camada Gold (dims + fcts) e os resultados das queries analíticas (Q1–Q4) para CSV local, prontos para importação no Power BI Desktop.

```
data/gold_exports/
├── tables/     → dim_time.csv, dim_plan.csv, fct_plan_premium.csv, ...
└── analytics/  → q1_evolucao_copay_coinsurance.csv, q2_..., ...
```

## Como executar

1. Abra o projeto no Dev Container (`Ctrl+Shift+P` → **Dev Containers: Reopen in Container**).
2. Atualize as credenciais via task **Refresh AWS Credentials** se a sessão do Learner Lab tiver expirado.
3. Selecione o kernel **Python 3.11.14**.
4. Execute as células sequencialmente (`Run All`).

> **No Power BI Desktop:** Obter Dados → Texto/CSV → importe os arquivos de `data/gold_exports/`.
> Use **Import Mode** (não DirectQuery) para evitar consultas recorrentes ao Athena.

---
## 1. Configuração

In [ ]:
import awswrangler as wr
import boto3
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

In [ ]:
account_id = boto3.client("sts").get_caller_identity()["Account"]

GOLD_DATABASE    = "eedb015_gold"
S3_ATHENA_OUTPUT = f"s3://{account_id}-eedb015-athena-results/"

NOTEBOOK_DIR  = Path().resolve()          # scripts/
PROJECT_ROOT  = NOTEBOOK_DIR.parent
TABLES_DIR    = PROJECT_ROOT / "data" / "gold_exports" / "tables"
ANALYTICS_DIR = PROJECT_ROOT / "data" / "gold_exports" / "analytics"
SQL_DIR       = PROJECT_ROOT / "src" / ".sql" / "analytics"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
ANALYTICS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Account ID    : {account_id}")
print(f"Database      : {GOLD_DATABASE}")
print(f"Athena output : {S3_ATHENA_OUTPUT}")
print(f"Saída local   : {PROJECT_ROOT / 'data' / 'gold_exports'}")

---
## 2. Exportação das Tabelas (dims + fcts)

Exporta todas as tabelas da camada Gold sem filtro.

In [ ]:
TABLES = [
    # Dimensões
    "dim_time",
    "dim_geography",
    "dim_benefit_category",
    "dim_issuer",
    "dim_network",
    "dim_plan",
    # Fatos
    "fct_market_competition",
    "fct_benefit_coverage",
    "fct_plan_premium",
]


def export_table(table_name: str) -> pd.DataFrame:
    sql = f"SELECT * FROM {GOLD_DATABASE}.{table_name}"
    print(f"↓ {table_name}...", end=" ", flush=True)
    df = wr.athena.read_sql_query(
        sql=sql,
        database=GOLD_DATABASE,
        s3_output=S3_ATHENA_OUTPUT,
        ctas_approach=False,
    )
    dest = TABLES_DIR / f"{table_name}.csv"
    df.to_csv(dest, index=False)
    print(f"{len(df):,} linhas  →  {dest.name}")
    return df


for table in TABLES:
    export_table(table)

---
## 3. Queries Analíticas (Q1–Q4)

Descobre e executa automaticamente todos os arquivos `.sql` em `src/.sql/analytics/`.

In [ ]:
def export_query(sql_file: Path) -> pd.DataFrame:
    question = sql_file.parent.name   # "q1"
    label    = f"{question}_{sql_file.stem}"  # "q1_evolucao_copay_coinsurance"
    print(f"↓ {label}...", end=" ", flush=True)
    sql = sql_file.read_text(encoding="utf-8")
    df = wr.athena.read_sql_query(
        sql=sql,
        database=GOLD_DATABASE,
        s3_output=S3_ATHENA_OUTPUT,
        ctas_approach=False,
    )
    dest = ANALYTICS_DIR / f"{label}.csv"
    df.to_csv(dest, index=False)
    print(f"{len(df):,} linhas  →  {dest.name}")
    return df


for sql_file in sorted(SQL_DIR.rglob("*.sql")):
    export_query(sql_file)

---
## 4. Resumo

In [ ]:
csvs = sorted((PROJECT_ROOT / "data" / "gold_exports").rglob("*.csv"))
total_mb = sum(f.stat().st_size for f in csvs) / 1024 / 1024

print(f"Total: {len(csvs)} arquivo(s) — {total_mb:.1f} MB\n")
for csv in csvs:
    size_kb = csv.stat().st_size / 1024
    print(f"  {csv.relative_to(PROJECT_ROOT)}  ({size_kb:.0f} KB)")

print("\nPróximos passos no Power BI:")
print("  1. Power BI Desktop → Obter Dados → Texto/CSV")
print("  2. Importe os arquivos de data/gold_exports/")
print("  3. Use Import Mode (não DirectQuery)")